# 04.2 · kv-transfer (Spec-2) — data pipeline demo

`SRC.task != DST.task`, `SRC.progress == DST.progress`. Two distinct tasks that share a **common progress** (a prerequisite sub-computation) are clustered together; both prefill the same progress code. Transplant the SRC **task-description key+value** onto DST (GQA-corrected headwise routing) → does DST implement SRC's task?

Also demonstrates **cross-language** transfer: cache the task in Python, trigger it in C++.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-10-15 22:41:04 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
import json
with open(
    os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR), "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")
) as f:
    GOAL_HEADS_250 = [(layer_idx, head_idx) for layer_idx, head_idx in json.load(f)]

## Intervention knobs

`ALPHA` — OV-write boost of `PATCH_HEADS` at the pivot during the **patched** gen/readout (`1.0` = off) — the only knob; every intervention here is a **span** k+v transplant (no pivot-query window). Default is a no-op.

In [3]:
from contextlib import nullcontext

#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
QUERY_POSITIONS = "last"  # which DST query positions of PATCH_HEADS read the transplanted span:
                         #   "all"  = every query after the span (original; pre-pivot latents change too)
                         #   "last" = only the last query of each forward (pivot, then each generated token)
                         #            -> pre-pivot DST latents are untouched
                         #   "last_n" = the last QUERY_LAST_N prompt tokens (ending at the pivot), then each
                         #            generated token -> latents before them are untouched
QUERY_LAST_N = 4         # only used when QUERY_POSITIONS == "last_n" (1 == "last")
QP = QUERY_LAST_N if QUERY_POSITIONS == "last_n" else QUERY_POSITIONS   # what headwise_kv_patch receives
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
READOUT_HEADS = GOAL_HEADS                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 43)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------

def ov_boost(heads=None, alpha=None, positions=(-1,)):
    """Context manager scaling the OV write of `heads` by `alpha` (defaults PATCH_HEADS/ALPHA);
    alpha==1.0 -> nullcontext (no-op). Stack it under `headwise_kv_patch` for the patched gen."""
    heads = PATCH_HEADS if heads is None else heads
    alpha = ALPHA if alpha is None else alpha
    return boost_head_ov(mt, heads, alpha, positions=positions) if alpha != 1.0 else nullcontext()



def ablate_ctx(heads=None, keep=None, block_span=None):
    """Ablate `heads` (default ABLATE_HEADS) on the PATCHED run; `keep` heads (default PATCH_HEADS)
    are never ablated so a patched head is never touched. Strategy per ABLATION_STRAT:
      "zero"     -> kill their OV write (boost_head_ov x0.0 at ALL positions);
      "restrict" -> attention-knockout: block their attention to `block_span` (the DST <task_span>)
                    so they can look everywhere else. "restrict" requires `block_span=(start, end)`."""
    heads = ABLATE_HEADS if heads is None else heads
    keep = set(PATCH_HEADS if keep is None else keep)
    heads = [h for h in heads if h not in keep]
    if not heads:
        return nullcontext()
    if ABLATION_STRAT == "restrict":
        assert block_span is not None, "restrict ablation needs block_span (the DST task span)"
        return restrict_head_attn(mt, heads, range(block_span[0], block_span[1]), keep=keep)
    return boost_head_ov(mt, heads, 0.0, positions=None)

def headwise_iface(patch_heads, k_cache, v_cache, dst_pos, alpha=None, query_positions=None):
    """headwise_kv_patch patch_interface, composed with the ALPHA OV-boost — for the PATCHED
    goalscope readout. At ALPHA==1.0 it matches the plain `make_headwise_interface` iface.
    `query_positions` defaults to QP (read at call time)."""
    def _if(mt, inputs, patches=None, model_kwargs=None):
        qp = QP if query_positions is None else query_positions
        with headwise_kv_patch(mt, patch_heads, k_cache, v_cache, dst_pos, query_positions=qp), ov_boost(patch_heads, alpha):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


# ── two-span prompt renderer: full task span (bright_yellow) + the EXACT KV window (orange) ────
# The transplant window is chosen by `align_kv_spans(mt, ..., mode=ALIGN_KNOB)` (src.diff.functional).
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Render the prompt exactly as the model sees it. task_span -> bright_yellow (the task text);
    kv_span -> `black on dark_orange` (the exact tokens whose k+v are transplanted); pivot -> salmon.
    Priority pivot > kv > task, so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids)
    pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    # Mark spans with control-char sentinels and escape the WHOLE joined string once (then swap
    # sentinels for tags). Escaping per-token drops brackets like `[w ...]` / `list[str]` when the
    # tokenizer splits `[` from the following letter across two tokens: the lone `[` isn't escaped,
    # so once joined it reads as a markup tag and rich absorbs it.
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Inspect the generated clusters / setups

In [4]:
for dt, clusters in D.KVCLUSTERS.items():
    print(f"{dt}:")
    for cl in clusters:
        print(f"  cluster {cl.id!r}: prereq={cl.common_progress.result_name!r} "
              f"({cl.common_progress.description!r}) · tasks={[t.id for t in cl.tasks]}")
print()
setups = D.kv_transfer_setups("list[str]", presentation="description")
print(f"list[str]: {len(setups)} ordered pairs across its clusters")
for s in setups[:12]:
    print(f"  [{s.meta['cluster']:<14}] src={s.src_task:<22} dst={s.dst_task:<22} pivot={s.pivot}")

list[str]:
  cluster 'sorted_by_len': prereq='sorted_words' ('the words sorted by length') · tasks=['shortest_vowel_start', 'longest_palindrome', 'longest_len', 'shortest_len', 'median_word_len', 'longest_starts_upper']
  cluster 'long_words': prereq='long_words' ('the words that have at least two characters') · tasks=['count_palindromes', 'count_all_distinct', 'count_same_ends']
  cluster 'lowercased': prereq='lowered' ('the words converted to lowercase') · tasks=['count_ci_palindromes', 'num_distinct_ci', 'count_contains_a', 'any_ci_duplicate']
list[int]:
  cluster 'sorted_asc': prereq='sorted_nums' ('the values sorted in ascending order') · tasks=['median_val', 'range_span', 'second_largest', 'second_smallest', 'max_gap']
  cluster 'positives': prereq='positives' ('the values that are greater than zero') · tasks=['sum_positives', 'count_positives', 'product_positives', 'smallest_positive', 'all_even_positives']
  cluster 'prefix_sums': prereq='prefix' ('the running cumulative sums o

## Render one setup — SRC/DST prompts with the transplanted task span marked

In [5]:
inst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="cpp", dst_lang="python",
)[4]
m = D.materialize(inst)
SRC_SPAN0 = token_span(m["src_prompt"], m["src_task_text"])   # located target spans (pre-align)
DST_SPAN0 = token_span(m["dst_prompt"], m["dst_task_text"])
print(f"SRC task = {m['src_task_text']!r}\nDST task = {m['dst_task_text']!r}")
# align -> TokenizerOutputs (pad token-splices \n before the shorter target). SRC_POS = cached-from,
# DST_POS = patched-to. For pad both spans are K=max -> the whole span is orange (fully patched).
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], SRC_SPAN0, DST_SPAN0, mode=ALIGN_KNOB,
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
piv_src, piv_dst = src_tok["input_ids"].shape[1] - 1, dst_tok["input_ids"].shape[1] - 1
show_spans(src_tok, f"SRC · CACHED-FROM {tuple(SRC_SPAN)} (orange) · pivot@{piv_src} · align={ALIGN_KNOB}",
           task_span=SRC_SPAN, kv_span=(SRC_POS[0], SRC_POS[-1] + 1), piv=-1)
show_spans(dst_tok, f"DST · PATCHED-TO {tuple(DST_SPAN)} (orange) · pivot@{piv_dst} · align={ALIGN_KNOB}",
           task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), piv=-1)

SRC task = 'the length of the median word when the words are ordered by length'
DST task = 'the length of the shortest word'


╭─ SRC · CACHED-FROM (64, 77) (orange) · pivot@158 · align=pad ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ the length of the median word when the words are ordered by length\n                                            │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::vector<std::string> sorted_words = words;\n                                                            │
│     std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){  │
│ return a.size() < b.size(); });\n                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · PATCHED-TO (63, 76) (orange) · pivot@103 · align=pad ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ the length of the shortest word\n                                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant SRC task-description key+value onto DST

Align the two task spans (per `ALIGN_KNOB` — right-edge or pad), cache SRC key+value over `PATCH_HEADS`, and run the destination under `headwise_kv_patch` (only `PATCH_HEADS` read the source span — GQA bleed corrected). Expect patched DST to implement SRC's task.

In [6]:
# ── intervention: transplant SRC task-description k+v onto DST — grouped per run ──
from src.diff.functional import _layers_of, heads_per_layer, cache_src_proj

# ALPHA = 3.0
# PATCH_HEADS = UNION_HEADS

# src_tok / dst_tok / SRC_POS (cached-from) / DST_POS (patched-to) / piv come from the render cell above
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "k")
v_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "v")

# ── source_run — clean SRC ──
gen_src = greedy(src_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC ({m['src_task_text']!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_tok, piv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST ──
gen_dst = greedy(dst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST ({m['dst_task_text']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_tok, piv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — SRC task k+v transplanted (+ ALPHA OV-boost) · QUERY_POSITIONS: all / last queries ──
_boost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS, query_positions=QP), _boost, ablate_ctx(keep=PATCH_HEADS, block_span=DST_SPAN):
    _patched = greedy(dst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {m['src_task_text']!r}):[/salmon1] \n{escape(_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_tok, piv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, k_cache, v_cache, DST_POS))[0]["patched_gen"]

# ── eval pipeline — causal + readout (no gaze) ──
from src.diff.eval import eval_kv_transfer
_ev = eval_kv_transfer(inst, patched_gen=_patched, readouts=(ro_src, ro_dst, ro_patched))
rich_print(f"[bold]eval · kv-cluster[/bold] : {_ev.summary()}")

source_run · clean SRC ('the length of the median word when the words are ordered by length'): 
 size_t median_index = sorted_words.size() / 2;
     return sorted_words[median_index].size();
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=158 '    ' => I'm identifying the middle element of a sorted list or array.

destination_run · clean DST ('the length of the shortest word'): 
 len(sorted_words[0])
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=103 ' return' => I'm determining the length of the shortest string in the 
list by using the `len()` function on the first element after sorting the list by string length.

patched_run · DST (expect 'the length of the median word when the words are ordered by length'): 
 len(sorted_words[0]) if sorted_words else 0
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=103 ' return' => I'm calculating the length of the shortest string in a list by 
sorting the list by string length and returning the length of the first element.

eval · kv-cluster : causal: no (dst) | readout: src→dst→dst

### Check — do the pre-pivot DST latents change under the transplant?

Residual stream (every layer output) of the clean DST run vs the patched run (k+v transplant + ALPHA boost, no
ablation): max |Δ| over the pre-pivot positions after the task span, and at the pivot. `"last"` / an int n should give exactly 0
before the routed queries (the pivot / the last n prompt tokens); `"all"` should not (there every pre-pivot
position after the span is routed).

In [7]:
import baukit
from contextlib import ExitStack


def residuals(tok, ctx_fn=nullcontext):
    names = [mt.layer_name_format.format(l) for l in range(mt.n_layer)]
    with torch.no_grad(), ctx_fn(), baukit.TraceDict(mt._model, names, retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    out = lambda o: (o[0] if isinstance(o, tuple) else o)
    return torch.stack([out(tr[n].output)[0].float().cpu() for n in names])   # (L, T, d)


def patched_ctx(qp):
    def _c():
        s = ExitStack()
        s.enter_context(headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS, query_positions=qp))
        s.enter_context(ov_boost(PATCH_HEADS, ALPHA))
        return s
    return _c


H_dst = residuals(dst_tok)
span_end, piv = DST_SPAN[1], dst_tok["input_ids"].shape[1] - 1
for qp in dict.fromkeys(("all", "last", QP)):  # + the knob's value
    d = (residuals(dst_tok, patched_ctx(qp)) - H_dst).abs()                    # (L, T, d)
    lo = piv if qp == "all" else piv - (1 if qp == "last" else qp) + 1          # first routed query (all: n/a)
    end = piv if qp == "all" else lo                                            # region that SHOULD be untouched
    pre = d[:, span_end:end].amax(dim=(1, 2)) if end > span_end else torch.zeros(1)
    at_piv = d[:, piv].amax(dim=-1)
    first = int((pre > 0).nonzero()[0]) if (pre > 0).any() else None
    rich_print(f"[bold]{str(qp):>4}[/bold] · max|Δ| after span, before the routed queries ({end - span_end} pos) = "
               f"{pre.max():.4g} (first layer ≠ 0: {first}) · at pivot = {at_piv.max():.4g}")

 all · max|Δ| after span, before the routed queries (27 pos) = 360 (first layer ≠ 0: 28) · at pivot = 61

last · max|Δ| after span, before the routed queries (27 pos) = 0 (first layer ≠ 0: None) · at pivot = 112

## Cross-language — cache the task in Python, trigger it in a C++ destination

In [8]:
# ── cross-language: cache the task in Python, trigger it in a C++ destination — grouped per run ──
xinst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="cpp", dst_lang="python"
)[3]
xm = D.materialize(xinst)
XS0 = token_span(xm["src_prompt"], xm["src_task_text"])
XD0 = token_span(xm["dst_prompt"], xm["dst_task_text"])
xsrc_tok, xdst_tok, XSP, XDP, XS, XD = align_kv_spans(
    mt, xm["src_prompt"], xm["dst_prompt"], XS0, XD0, mode=ALIGN_KNOB,
    src_text=xm["src_task_text"], dst_text=xm["dst_task_text"])
xpiv_src, xpiv_dst = xsrc_tok["input_ids"].shape[1] - 1, xdst_tok["input_ids"].shape[1] - 1
show_spans(xsrc_tok, f"SRC (python) · CACHED-FROM {tuple(XS)} (orange) · align={ALIGN_KNOB}",
           task_span=XS, kv_span=(XSP[0], XSP[-1] + 1), piv=-1)
show_spans(xdst_tok, f"DST (c++)    · PATCHED-TO {tuple(XD)} (orange) · align={ALIGN_KNOB}",
           task_span=XD, kv_span=(XDP[0], XDP[-1] + 1), piv=-1)

xkvh = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
xk = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "k")
xv = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "v")

# ── source_run (python) ──
xgen_src = greedy(xsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC/python ({xm['src_task_text']!r}):[/red1] \n{escape(xgen_src)}")
xro_src = goalscope_readout("   ↳ goalscope · source_run", xsrc_tok, xpiv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run (c++) ──
xgen_dst = greedy(xdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST/c++ ({xm['dst_task_text']!r}):[/sea_green3] \n{escape(xgen_dst)}")
xro_dst = goalscope_readout("   ↳ goalscope · destination_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — python task k+v -> c++ DST (expect the PYTHON src task) ──
_xboost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, xk, xv, XDP, query_positions=QP), _xboost, ablate_ctx(keep=PATCH_HEADS, block_span=XD):
    xpatched = greedy(xdst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST/c++ (expect {xm['src_task_text']!r}):[/salmon1] \n{escape(xpatched)}")
xro_patched = goalscope_readout("   ↳ goalscope · patched_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, xk, xv, XDP))[0]["patched_gen"]

from src.diff.eval import eval_kv_transfer
_xev = eval_kv_transfer(xinst, patched_gen=xpatched, readouts=(xro_src, xro_dst, xro_patched))
rich_print(f"[bold]eval · kv-cluster (cross-lang)[/bold] : {_xev.summary()}")

╭─ SRC (python) · CACHED-FROM (64, 74) (orange) · align=pad ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ returns true if the shortest word starts with a vowel\n                                                         │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::vector<std::string> sorted_words = words;\n                                                            │
│     std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){  │
│ return a.size() < b.size(); });\n                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST (c++)    · PATCHED-TO (63, 73) (orange) · align=pad ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ returns true if the longest word is a palindrome\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

source_run · clean SRC/python ('returns true if the shortest word starts with a vowel'): 
 char first_char = std::tolower(sorted_words[0][0]);
    return first_char == 'a' || first_char == 'e' || first_char == 'i' || first_char == 'o' || first_char == 'u';
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=155 '    ' => I'm checking if the first character of the string is a vowel.

destination_run · clean DST/c++ ('returns true if the longest word is a palindrome'): 
 sorted_words[-1] == sorted_words[-1][::-1]
</code>

... (2 outputs left out)

patched_run · DST/c++ (expect 'returns true if the shortest word starts with a vowel'): 
 sorted_words[0][0].lower() in 'aeiou'
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=100 ' return' => I'm checking if the first character of the string is a vowel.

eval · kv-cluster (cross-lang) : causal: TRIGGERED (src) | readout: src→dst→src

In [9]:
print(mt.tokenizer.decode(src_tok.input_ids[0].tolist()))

<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
the length of the median word when the words are ordered by length
Assume that all the libraries have been already imported.
<|im_end|>
<|im_start|>assistant
<code>
auto func(std::vector<std::string> words){
    std::vector<std::string> sorted_words = words;
    std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){ return a.size() < b.size(); });
    


## Multi-predicate setup — `SRC(T1,T2,T3)` vs `DST(T1,T2,T4)` (kv-transfer)

A kv-transfer variant purpose-built for this notebook. SRC and DST share a **prefilled prefix** (T1, T2 assigned identically) and diverge only in the **frontier** predicate their *question* lists — SRC asks for T3, DST for T4. The frontier description is a distinct span (T3 in SRC's question, T4 in DST's), so this is a genuine **k+v transplant** (not query): cache the SRC **T3-description** key+value and transplant it onto the DST **T4-description** span. Test: does patched DST implement **T3** instead of T4? Gaze is **N/A** (T3 never appears in the DST prompt); eval = causal + readout.

In [10]:
ALIGN_KNOB = "pad"
# PATCH_HEADS = UNION_HEADS
ALPHA = 2.0
PATCH_HEADS = [(layer_idx, head_idx) for layer_idx in range(35, 50) for head_idx in range(mt.n_heads)]
len(PATCH_HEADS)

600

In [107]:
mp = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=0,
    src_lang="cpp", dst_lang="python", naming="letters", letter_stub="var_eq"
)
for s in mp:
    print(f"  shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<16} DST.frontier={s.dst_next}")

inst_mp = mp[7]
mm = D.materialize(inst_mp)
MP_SRC0 = token_span(mm["src_prompt"], mm["src_task_text"])   # T3 description (pre-align)
MP_DST0 = token_span(mm["dst_prompt"], mm["dst_task_text"])   # T4 description (pre-align)
print(f"\nSRC frontier (T3) = {mm['src_task_text']!r}")
print(f"DST frontier (T4) = {mm['dst_task_text']!r}")
print(f"shared (T1,T2)    = {inst_mp.task_desc!r}")
mp_src_tok, mp_dst_tok, MP_SRC_POS, MP_DST_POS, MP_SRC_SPAN, MP_DST_SPAN = align_kv_spans(
    mt, mm["src_prompt"], mm["dst_prompt"], MP_SRC0, MP_DST0, mode=ALIGN_KNOB,
    src_text=mm["src_task_text"], dst_text=mm["dst_task_text"])
show_spans(mp_src_tok, f"SRC · T3 CACHED-FROM {tuple(MP_SRC_SPAN)} (orange) · pivot@{mp_src_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_SRC_SPAN, kv_span=(MP_SRC_POS[0], MP_SRC_POS[-1] + 1), piv=-1)
show_spans(mp_dst_tok, f"DST · T4 PATCHED-TO {tuple(MP_DST_SPAN)} (orange) · pivot@{mp_dst_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_DST_SPAN, kv_span=(MP_DST_POS[0], MP_DST_POS[-1] + 1), piv=-1)

  shared=['all_unique_lengths', 'has_palindrome']  SRC.frontier=distinct_count   DST.frontier=count_short
  shared=['any_empty', 'count_ends_consonant']  SRC.frontier=total_uppercase_letters DST.frontier=distinct_count
  shared=['total_vowels', 'longest_word_length']  SRC.frontier=count_capitalized DST.frontier=count_short
  shared=['longest_word_length', 'count_long']  SRC.frontier=any_empty        DST.frontier=total_length
  shared=['distinct_count', 'total_uppercase_letters']  SRC.frontier=total_length     DST.frontier=total_vowels
  shared=['shortest_word_length', 'count_all_lowercase']  SRC.frontier=count_short      DST.frontier=total_length
  shared=['all_unique_lengths', 'count_starts_vowel']  SRC.frontier=longest_word_length DST.frontier=shortest_word_length
  shared=['total_uppercase_letters', 'count_all_lowercase']  SRC.frontier=all_unique_lengths DST.frontier=count_short
  shared=['total_length', 'any_empty']  SRC.frontier=count_short      DST.frontier=has_palindrome
  share

╭─ SRC · T3 CACHED-FROM (64, 101) (orange) · pivot@251 · align=pad ───────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the total number of uppercase letters across all words\n                                                    │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) whether all words have distinct lengths\n                                                                   │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     auto a = 0;\n                                                                                               │
│     for(auto &a_w : words) for(char a_c : a_w) if(std::isupper((unsigned char)a_c)) a++;\n                      │
│     auto b = 0;\n                                                                                               │
│     for(auto &b_w : words){ bool b_has = false, b_ok = true; for(char b_c : b_w){ if(std::isalpha((unsigned     │
│ char)b_c)){ b_has = true; if(std::isupper((unsigned char)b_c)) b_ok = false; } } if(b_has && b_ok) b++; }\n     │
│     auto c =                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · T4 PATCHED-TO (63, 100) (orange) · pivot@152 · align=pad ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the total number of uppercase letters across all words\n                                                    │
│ (2) the number of words written entirely in lowercase letters\n                                                 │
│ (3) the number of words shorter than three characters\n                                                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     b = sum(1 for w in words if w.islower())\n                                                                  │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [76]:
mp_dst_tok.input_ids.shape[1], MP_DST_SPAN, mp_dst_tok.input_ids.shape[1] - MP_DST_SPAN[0]

(153, (63, 100), 90)

## Multi-predicate intervention — k+v transplant of the frontier-predicate span

Same `run_kv_transfer_intervention` as the kv-cluster (align the T3/T4 spans per `ALIGN_KNOB`, cache SRC key+value over `PATCH_HEADS`, GQA-corrected headwise routing), scored with `eval_kv_transfer` (causal + readout; no gaze). `ALPHA` applies.

In [105]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
QUERY_POSITIONS = "all"  # which DST query positions of PATCH_HEADS read the transplanted span:
                         #   "all"  = every query after the span (original; pre-pivot latents change too)
                         #   "last" = only the last query of each forward (pivot, then each generated token)
                         #            -> pre-pivot DST latents are untouched
                         #   "last_n" = the last QUERY_LAST_N prompt tokens (ending at the pivot), then each
                         #            generated token -> latents before them are untouched
QUERY_LAST_N = 4         # only used when QUERY_POSITIONS == "last_n" (1 == "last")
QP = QUERY_LAST_N if QUERY_POSITIONS == "last_n" else QUERY_POSITIONS   # what headwise_kv_patch receives
# PATCH_HEADS = GOAL_HEADS_250                    # heads carrying the transplant (GOAL | UNION | ...)
PATCH_HEADS = [(layer_idx, head_idx) for (layer_idx, head_idx) in GOAL_HEADS_250 if layer_idx>=30 or layer_idx==2]
# PATCH_HEADS = [(layer_idx, head_idx) for layer_idx in range(40, 50) for head_idx in range(mt.n_heads)]
READOUT_HEADS = GOAL_HEADS_250                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 64)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------
print(f"{len(PATCH_HEADS)=}")

len(PATCH_HEADS)=230


In [106]:
from src.diff.functional import run_kv_transfer_intervention
from src.diff.eval import eval_kv_transfer


def report(tag, inst, out):
    ev = eval_kv_transfer(
        inst,
        patched_gen=out["gen_patched"],
        readouts=(out["ro_src"], out["ro_dst"], out["ro_patched"]),
    )
    rich_print(
        f"[bold]{tag}[/bold]  SRC.target={inst.expected_next_desc!r}  DST.target={inst.dst_next_desc!r}"
    )
    rich_print(f"  [red1]clean SRC[/red1]    :\n{escape(out['gen_src'])}")
    rich_print(f"  [sea_green3]clean DST[/sea_green3]    :\n{escape(out['gen_dst'])}")
    rich_print(f"  [salmon1]patched DST[/salmon1]  :\n{escape(out['gen_patched'])}")
    rich_print(
        f"  readout src→dst→pat : {out['ro_src']!r} → {out['ro_dst']!r} → {out['ro_patched']!r}"
    )
    rich_print(f"  [bold]eval[/bold] : {ev.summary()}")
    return ev


# k+v transplant of the SRC frontier (T3) description onto the DST frontier (T4) span
report(
    f"multi-predicate · k+v transplant · {inst_mp.meta['datatype']} · align={ALIGN_KNOB}",
    inst_mp,
    run_kv_transfer_intervention(
        mt=mt,
        inst=inst_mp,
        patch_heads=PATCH_HEADS,
        # patch_heads=GOAL_HEADS_250,
        readout_heads=READOUT_HEADS,
        alpha=ALPHA,
        align=ALIGN_KNOB,
        query_positions=QP,
        # ablate_heads=ABLATE_HEADS,
        # ablation_strat=ABLATION_STRAT,
    ),
)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=251 ' =' => Checking for unique elements using a set.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=152 ' =' => I'm checking if the length of the string is less than 3.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=152 ' =' => I'm checking if the current line is empty.

... (3 outputs left out)

patched DST  :
 sum(1 for w in words if len(w) == 1)
    return a, b, c
</code>

readout src→dst→pat : 'Checking for unique elements using a set.' → "I'm checking if the length of the string is 
less than 3." → "I'm checking if the current line is empty."

eval : causal: no (dst) | readout: src→dst→neither

QTransferEval(causal={'label': 'dst', 'why': 'The computation counts words where len(w) == 1, which is a partial implementation of counting words shorter than three characters (sub-goal B), not checking for distinct lengths (sub-goal A).', 'raw': 'The first computation is `sum(1 for w in words if len(w) == 1)`, which counts words with exactly 1 character (length < 3, but specifically length == 1). This is a partial implementation of counting words shorter than 3 characters, though it only counts words of length 1, not length 1 or 2.\n\nHowever, the task asks me to judge by what computation is actually performed. The computation checks `len(w) == 1`, which counts words of length exactly 1. This is a subset of "words shorter than three characters" (which would be `len(w) < 3`). It\'s a partial/approximate implementation of sub-goal B.\n\nSub-goal A would require checking if all words have distinct lengths, which this does not do.\n\n<verdict>B</verdict>\n<why>The computation counts words

## Residual transfer of the shared code prefix — `p_src: a'b'c -> c` vs `p_dst: a'b'd -> d` (py → py)

SRC lists (a, b, c), DST lists (a, b, d); both prefill `a` and `b` and stop at the frontier's `c =`. Everything from the
user turn's `<|im_end|>` to the ` =` pivot is **token-identical**:
```
<|im_end|>\n<|im_start|>assistant\n<code>\ndef func(words: list[str]):\n    a = ...\n    b = ...\n    c =
```
Over this segment, the SRC residual stream (output of **every** decoder layer, `RP_LAYERS`) replaces DST's. The DST task
list (`d`) is untouched; everything after the user turn now carries SRC's computation. Question: does the patched DST write
SRC's `c` (the goal rides in the prefix latents) or its own `d` (it is re-derived from the task span)?

Caveats:
* With the pivot ` =` included (`RP_INCLUDE_PIVOT=True`) and the last layer patched, DST's next-token logits are SRC's
  **by construction**, so the first generated token is copied. Only the tokens after it are informative. `False` leaves
  the pivot clean: it then reads DST's task span and the SRC-patched k/v of the segment.
* `RP_INCLUDE_SRC_FRONT=True` also transfers the frontier-description span (SRC's `c` text -> DST's `d` text). If the two
  differ in token length, the shorter one is padded with `\n` tokens right before it (`align_kv_spans(mode="pad")`), so
  every token of DST's `d` span is overwritten. All three runs (clean SRC, clean DST, patched) then use the padded prompts.
* Generated tokens are not patched; they read the KV cache, whose segment k/v come from the patched residuals.
* The GoalScope readout at ` =` reads the goal heads' write at the pivot under the patch. Under `RP_INCLUDE_PIVOT=True` their
  input is SRC's residual (layer l-1) while their keys still include DST's task span.

In [131]:
#------------------------------------------------------------------------------------------------------
RP_SEED, RP_IDX = 0, 4          # python->python instance: multi_predicate_setups(seed=RP_SEED)[RP_IDX]
RP_LAYERS = "all"               # "all" or an iterable of layer indices whose OUTPUT residual is transferred
RP_INCLUDE_PIVOT = False        # False: transfer UNTIL the pivot ` =` (pivot left clean) | True: also the pivot (then the first generated token is SRC's by construction)
RP_INCLUDE_SRC_FRONT = False    # True: also transfer the frontier description (SRC c-span -> DST d-span), "\n"-padded to equal length
RP_MAX_NEW = 64
RP_ORACLE = True                # eval_kv_transfer (causal + readout oracle) on the patched run
#------------------------------------------------------------------------------------------------------
import baukit
from contextlib import contextmanager


def layer_outputs(tok, positions, layers):
    """{layer: (len(positions), d)} — each decoder layer's OUTPUT residual at `positions` (CPU float32)."""
    names = {l: mt.layer_name_format.format(l) for l in layers}
    with torch.no_grad(), baukit.TraceDict(mt._model, list(names.values()), retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    out = lambda o: o[0] if isinstance(o, tuple) else o
    return {l: out(tr[n].output)[0, positions].detach().float().cpu() for l, n in names.items()}


@contextmanager
def residual_patch(cache, positions, layers):
    """Overwrite each decoder layer's OUTPUT residual at absolute `positions` with `cache[layer]`.
    Only forwards that contain those positions are touched (the prompt pass); decode steps (1 token)
    are skipped — the effect then rides the KV cache, whose k/v at `positions` were computed from the
    patched residuals."""
    handles = []

    def make(src):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            if h.shape[1] <= max(positions):
                return out
            h = h.clone()
            h[:, positions] = src.to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        return hook

    try:
        for l in layers:
            mod = baukit.get_module(mt._model, mt.layer_name_format.format(l))
            handles.append(mod.register_forward_hook(make(cache[l])))
        yield
    finally:
        for hd in handles:
            hd.remove()


def show_windows(tok, title, task_span=None, windows=(), mark_pivot=True):
    """The prompt as the model sees it: task_span -> yellow, every transferred window -> orange,
    pivot -> salmon (only if `mark_pivot`). Text objects, so no markup escaping issues."""
    ids = tok["input_ids"][0].tolist()
    task = set(range(*task_span)) if task_span else set()
    win = {q for lo, hi in windows for q in range(lo, hi)}
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid]).replace("\n", "\\n\n")
        style = ("bold black on salmon1" if mark_pivot and i == len(ids) - 1
                 else "bold black on dark_orange" if i in win else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{escape(title)}[/bold]", title_align="left", border_style="grey42",
                     padding=(1, 1)))


In [132]:
mp_py = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=RP_SEED,
    src_lang="python", dst_lang="python", naming="letters", letter_stub="var_eq"
)
for i, s in enumerate(mp_py):
    print(f"[{i:2d}] shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<22} DST.frontier={s.dst_next}")

inst_rp = mp_py[RP_IDX]
mr = D.materialize(inst_rp)

# frontier-description spans = ONLY the frontier item's text (SRC c / DST d), not the whole task list. With
# RP_INCLUDE_SRC_FRONT the shorter one is "\n"-padded at the token level right before it, so both spans have equal
# length; the padded prompts are then used for ALL three runs.
RP_C_DESC, RP_D_DESC = inst_rp.src_frontier.description, inst_rp.dst_frontier.description
assert mr["src_prompt"].count(RP_C_DESC) == 1 and mr["dst_prompt"].count(RP_D_DESC) == 1, "frontier text not unique"
_fs0 = token_span(mr["src_prompt"], RP_C_DESC)
_fd0 = token_span(mr["dst_prompt"], RP_D_DESC)
if RP_INCLUDE_SRC_FRONT:
    rp_src_tok, rp_dst_tok, _, _, RP_SRC_FSPAN, RP_DST_FSPAN = align_kv_spans(
        mt, mr["src_prompt"], mr["dst_prompt"], _fs0, _fd0, mode="pad")
else:
    rp_src_tok = prepare_input(prompts=mr["src_prompt"], tokenizer=mt)
    rp_dst_tok = prepare_input(prompts=mr["dst_prompt"], tokenizer=mt)
    RP_SRC_FSPAN, RP_DST_FSPAN = _fs0, _fd0
src_ids, dst_ids = rp_src_tok["input_ids"][0].tolist(), rp_dst_tok["input_ids"][0].tolist()
L_src, L_dst = len(src_ids), len(dst_ids)

# shared segment = last <|im_end|> (end of the user turn) .. pivot, aligned from the END of each prompt
im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
seg_start = max(i for i, t in enumerate(dst_ids) if t == im_end)
n_seg = L_dst - seg_start
assert src_ids[L_src - n_seg:] == dst_ids[seg_start:], "SRC/DST code prefix is not token-identical"
n_pat = n_seg if RP_INCLUDE_PIVOT else n_seg - 1
seg_src = (L_src - n_seg, L_src - n_seg + n_pat)
seg_dst = (seg_start, seg_start + n_pat)

# transferred windows: [frontier span (pad-aligned)] + segment; SRC/DST positions pair up 1:1 in order
RP_SRC_WIN = ([RP_SRC_FSPAN] if RP_INCLUDE_SRC_FRONT else []) + [seg_src]
RP_DST_WIN = ([RP_DST_FSPAN] if RP_INCLUDE_SRC_FRONT else []) + [seg_dst]
RP_SRC_POS = [q for lo, hi in RP_SRC_WIN for q in range(lo, hi)]
RP_DST_POS = [q for lo, hi in RP_DST_WIN for q in range(lo, hi)]
assert len(RP_SRC_POS) == len(RP_DST_POS), (len(RP_SRC_POS), len(RP_DST_POS))
RP_LAYER_LIST = list(range(mt.n_layer)) if RP_LAYERS == "all" else sorted(RP_LAYERS)
RP_CFG = (RP_SEED, RP_IDX, RP_INCLUDE_PIVOT, RP_INCLUDE_SRC_FRONT)   # the run cell checks the knobs still match

_pad = (_fd0[1] - _fd0[0]) - (_fs0[1] - _fs0[0])
print(f"\nSRC frontier (c) = {RP_C_DESC!r}\nDST frontier (d) = {RP_D_DESC!r}")
print(f"frontier spans: SRC {_fs0[1] - _fs0[0]} tok, DST {_fd0[1] - _fd0[0]} tok"
      + (f" -> {abs(_pad)} '\\n' pad token(s) before the {'SRC' if _pad > 0 else 'DST'} span" if RP_INCLUDE_SRC_FRONT and _pad
         else "" if RP_INCLUDE_SRC_FRONT else " (not transferred)"))
print(f"segment: {n_seg} tokens · transferred {len(RP_DST_POS)} positions × {len(RP_LAYER_LIST)} layers · "
      f"windows SRC {RP_SRC_WIN} -> DST {RP_DST_WIN} · pivot {mt.tokenizer.decode([dst_ids[-1]])!r} "
      f"{'included' if RP_INCLUDE_PIVOT else 'left clean'}")
show_windows(rp_dst_tok, f"DST · task d (yellow) · transferred (orange) · L={L_dst}",
             task_span=RP_DST_FSPAN, windows=RP_DST_WIN, mark_pivot=RP_INCLUDE_PIVOT)

[ 0] shared=['all_unique_lengths', 'has_palindrome']  SRC.frontier=distinct_count         DST.frontier=count_short
[ 1] shared=['any_empty', 'count_ends_consonant']  SRC.frontier=total_uppercase_letters DST.frontier=distinct_count
[ 2] shared=['total_vowels', 'longest_word_length']  SRC.frontier=count_capitalized      DST.frontier=count_short
[ 3] shared=['longest_word_length', 'count_long']  SRC.frontier=any_empty              DST.frontier=total_length
[ 4] shared=['distinct_count', 'total_uppercase_letters']  SRC.frontier=total_length           DST.frontier=total_vowels
[ 5] shared=['shortest_word_length', 'count_all_lowercase']  SRC.frontier=count_short            DST.frontier=total_length
[ 6] shared=['all_unique_lengths', 'count_starts_vowel']  SRC.frontier=longest_word_length    DST.frontier=shortest_word_length
[ 7] shared=['total_uppercase_letters', 'count_all_lowercase']  SRC.frontier=all_unique_lengths     DST.frontier=count_short
[ 8] shared=['total_length', 'any_empty']  SR

╭─ DST · task d (yellow) · transferred (orange) · L=141 ──────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [133]:
# ── residual transfer: SRC segment residuals (every layer in RP_LAYER_LIST) -> DST ──
rp_cache = layer_outputs(rp_src_tok, RP_SRC_POS, RP_LAYER_LIST)
piv_s, piv_d = L_src - 1, L_dst - 1
assert RP_CFG == (RP_SEED, RP_IDX, RP_INCLUDE_PIVOT, RP_INCLUDE_SRC_FRONT), "knobs changed since the setup cell -- re-run it"

# both prompts: frontier task (yellow) · transferred windows (orange) · pivot (salmon, only if transferred)
show_windows(rp_src_tok, f"SRC · frontier c · CACHED-FROM {RP_SRC_WIN} · L={L_src}",
             task_span=RP_SRC_FSPAN, windows=RP_SRC_WIN, mark_pivot=RP_INCLUDE_PIVOT)
show_windows(rp_dst_tok, f"DST · frontier d · PATCHED-TO {RP_DST_WIN} · L={L_dst}",
             task_span=RP_DST_FSPAN, windows=RP_DST_WIN, mark_pivot=RP_INCLUDE_PIVOT)

rp_gen_src = greedy(rp_src_tok, max_new=RP_MAX_NEW)
rp_gen_dst = greedy(rp_dst_tok, max_new=RP_MAX_NEW)
with residual_patch(rp_cache, RP_DST_POS, RP_LAYER_LIST):
    rp_gen_patched = greedy(rp_dst_tok, max_new=RP_MAX_NEW)


def rp_iface(mt_, inputs, patches=None, model_kwargs=None):
    with residual_patch(rp_cache, RP_DST_POS, RP_LAYER_LIST):
        return mt_._model(**dict(inputs), **dict(model_kwargs or {}))


rp_ro_src = goalscope_readout("   ↳ goalscope · source_run", rp_src_tok, piv_s, heads=READOUT_HEADS)[0]["patched_gen"]
rp_ro_dst = goalscope_readout("   ↳ goalscope · destination_run", rp_dst_tok, piv_d, heads=READOUT_HEADS)[0]["patched_gen"]
rp_ro_patched = goalscope_readout("   ↳ goalscope · patched_run", rp_dst_tok, piv_d, heads=READOUT_HEADS,
                                  iface=rp_iface)[0]["patched_gen"]

for tag, col, gen, ro in (("clean SRC (c)", "red1", rp_gen_src, rp_ro_src),
                          ("clean DST (d)", "sea_green3", rp_gen_dst, rp_ro_dst),
                          ("patched DST", "salmon1", rp_gen_patched, rp_ro_patched)):
    rich_print(Panel(Text(gen), title=f"[{col}]{tag}[/{col}] · goalscope @ ' =': [italic]{escape(ro)}[/italic]",
                     title_align="left", border_style="grey42"))
print(f"SRC frontier (c) = {mr['src_task_text']!r}\nDST frontier (d) = {mr['dst_task_text']!r}")

if RP_ORACLE:
    from src.diff.eval import eval_kv_transfer
    _rev = eval_kv_transfer(inst_rp, patched_gen=rp_gen_patched, readouts=(rp_ro_src, rp_ro_dst, rp_ro_patched))
    rich_print(f"[bold]eval · residual transfer[/bold] : {_rev.summary()}")

╭─ SRC · frontier c · CACHED-FROM [(97, 140)] · L=141 ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of characters across all words\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · frontier d · PATCHED-TO [(97, 140)] · L=141 ─────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=140 ' =' => Counting characters.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=140 ' =' => I'm counting vowels in the string.

... (3 outputs left out)

╭─ clean DST (d) · goalscope @ ' =': I'm counting vowels in the string. ──────────────────────────────────────────╮
│  sum(1 for w in words for c in w if c.lower() in 'aeiou')                                                       │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST · goalscope @ ' =': I'm counting vowels in the string. ────────────────────────────────────────────╮
│  sum(1 for w in words for c in w if c.lower() in 'aeiou')                                                       │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC frontier (c) = '(1) the number of distinct words\n(2) the total number of uppercase letters across all words\n(3) the total number of characters across all words'
DST frontier (d) = '(1) the number of distinct words\n(2) the total number of uppercase letters across all words\n(3) the total number of vowels across all words'


eval · residual transfer : causal: no (dst) | readout: src→dst→dst

## Path patching — task-span k/v (I1) · progress restore (I2) · goal-head k/v from the pivot (I3) (py → py)

Same `a'b'c -> c` (SRC) vs `a'b'd -> d` (DST) instances as the residual-transfer section. One source k/v transplant over the
task span (`PP_SPAN`, `\n`-padded to equal length), read by two head routes, plus a residual restore. Each step has an
on/off knob:

* **I1 · `TASK_PATCH_HEADS`** read SRC's task-span key/value. Query rule `PP_I1_QUERY="all"` (as `QUERY_POSITIONS="all"`): every
  query after the span start, including the queries **inside** the task span, so DST's task-span latents change. Note that
  `"all"` also covers the pivot and every generated token, so these heads read SRC directly there as well.
* **I2 · progress restore.** I1 changes every later token. From `<|im_end|>` (`PP_I2_FROM="im_end"`) to the frontier variable
  `{var}` (pivot ` =` excluded), every decoder layer's output residual is reset to the **clean DST run**, so these positions'
  k/v are clean too. This **blocks** I1's path through the code progress. `"span_end"` also restores the few tokens between the
  task span and `<|im_end|>` (otherwise I1 can still reach the pivot through them).
* **I3 · `GOAL_PATCH_HEADS`** read SRC's task-span k/v **from the pivot on** (`PP_I3_QUERY="last"`: the pivot on the prompt pass,
  then each generated token). `PP_ALPHA` optionally boosts their OV write at `-1` (1.0 = off).

I1 and I3 share one transplant (`headwise_kv_patch_routes`); a head in both reads the source at the union of the two rules.
I2 acts on the prompt pass only; generated tokens read its effect through the KV cache. All three runs (clean SRC, clean DST,
patched) use the padded prompts. The GoalScope readout at ` =` runs under the same interventions.

In [ ]:
#------------------------------------------------------------------------------------------------------
PP_SEED, PP_IDX = 0, 0          # python->python instance: multi_predicate_setups(seed=PP_SEED)[PP_IDX]
PP_SPAN = "list"                # task span transplanted: "list" (whole task list = kv-transfer convention) | "frontier" (frontier item)
# I1 — task-span k/v
PP_I1 = True
TASK_PATCH_HEADS = GOAL_HEADS_250
PP_I1_QUERY = "all"             # "all" (spec: queries inside the span and after) | "last" | int n — see headwise_kv_patch
# I2 — restore clean-DST residuals over the code progress (pivot excluded)
PP_I2 = True
PP_I2_FROM = "im_end"           # "im_end" (from <|im_end|>) | "span_end" (from right after the task span)
PP_I2_LAYERS = "all"            # "all" or an iterable of layers whose OUTPUT residual is restored
# I3 — goal heads read the source k/v from the pivot on
PP_I3 = True
GOAL_PATCH_HEADS = GOAL_HEADS_250
PP_I3_QUERY = "last"            # "last" = pivot, then each generated token
PP_ALPHA = 1.0                  # OV boost of GOAL_PATCH_HEADS at -1 (pivot / each generated token) when PP_I3; 1.0 = off
PP_MAX_NEW = 64
PP_ORACLE = True                # eval_kv_transfer (causal + readout oracle) on the patched run
#------------------------------------------------------------------------------------------------------
from contextlib import ExitStack
from src.diff.functional import headwise_kv_patch_routes, heads_per_layer, cache_src_proj, boost_head_ov
assert "residual_patch" in globals() and "layer_outputs" in globals() and "show_windows" in globals(), \
    "run the residual-transfer knob cell first (defines layer_outputs / residual_patch / show_windows)"

In [ ]:
pp_mp = D.multi_predicate_setups(
    "list[str]", n_conditions=3, n_samples=16, seed=PP_SEED,
    src_lang="python", dst_lang="python", naming="letters", letter_stub="var_eq"
)
inst_pp = pp_mp[PP_IDX]
mp_ = D.materialize(inst_pp)
PP_C_DESC, PP_D_DESC = inst_pp.src_frontier.description, inst_pp.dst_frontier.description
_txt_s, _txt_d = ((mp_["src_task_text"], mp_["dst_task_text"]) if PP_SPAN == "list" else (PP_C_DESC, PP_D_DESC))
assert mp_["src_prompt"].count(_txt_s) == 1 and mp_["dst_prompt"].count(_txt_d) == 1, "task span text not unique"
_s0, _d0 = token_span(mp_["src_prompt"], _txt_s), token_span(mp_["dst_prompt"], _txt_d)
pp_src_tok, pp_dst_tok, PP_SRC_POS, PP_DST_POS, PP_SRC_SPAN, PP_DST_SPAN = align_kv_spans(
    mt, mp_["src_prompt"], mp_["dst_prompt"], _s0, _d0, mode="pad")
pp_dst_ids = pp_dst_tok["input_ids"][0].tolist()
PP_L_SRC, PP_L_DST = pp_src_tok["input_ids"].shape[1], len(pp_dst_ids)

# I2 window: <|im_end|> (or the end of the task span) .. {var}; the pivot ` =` (last token) is excluded
_im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
_i2_lo = (max(i for i, t in enumerate(pp_dst_ids) if t == _im_end) if PP_I2_FROM == "im_end" else PP_DST_SPAN[1])
PP_SEG = list(range(_i2_lo, PP_L_DST - 1))
PP_I2_LAYER_LIST = list(range(mt.n_layer)) if PP_I2_LAYERS == "all" else sorted(PP_I2_LAYERS)
PP_CFG = (PP_SEED, PP_IDX, PP_SPAN, PP_I2_FROM)   # the run cell checks these still match

print(f"SRC frontier (c) = {PP_C_DESC!r}\nDST frontier (d) = {PP_D_DESC!r}")
print(f"I1 task span ({PP_SPAN}): SRC {tuple(PP_SRC_SPAN)} -> DST {tuple(PP_DST_SPAN)} ({len(PP_DST_POS)} tok, pad-aligned) · "
      f"I2 restore: DST {PP_SEG[0]}..{PP_SEG[-1]} ({len(PP_SEG)} tok, "
      f"{mt.tokenizer.decode([pp_dst_ids[PP_SEG[0]]])!r} .. {mt.tokenizer.decode([pp_dst_ids[PP_SEG[-1]]])!r}) · "
      f"pivot {PP_L_DST - 1} {mt.tokenizer.decode([pp_dst_ids[-1]])!r}")
_gap = PP_SEG[0] - PP_DST_SPAN[1]
if _gap > 0:
    print(f"  not restored between the task span and I2: {_gap} tok {mt.tokenizer.decode(pp_dst_ids[PP_DST_SPAN[1]:PP_SEG[0]])!r}")
_ov = set(map(tuple, TASK_PATCH_HEADS)) & set(map(tuple, GOAL_PATCH_HEADS))
print(f"heads: TASK_PATCH_HEADS={len(TASK_PATCH_HEADS)} · GOAL_PATCH_HEADS={len(GOAL_PATCH_HEADS)} · in both: {len(_ov)}")
show_windows(pp_dst_tok, f"DST · I1 task span (orange) · I2 restored progress (yellow) · pivot (salmon) · L={PP_L_DST}",
             task_span=(PP_SEG[0], PP_SEG[-1] + 1), windows=[tuple(PP_DST_SPAN)], mark_pivot=True)

In [ ]:
# ── path patching: I1 (task-span k/v) + I2 (restore clean-DST progress) + I3 (goal heads read SRC from the pivot) ──
assert PP_CFG == (PP_SEED, PP_IDX, PP_SPAN, PP_I2_FROM), "knobs changed since the setup cell -- re-run it"
_routes = ([(TASK_PATCH_HEADS, PP_I1_QUERY)] if PP_I1 else []) + ([(GOAL_PATCH_HEADS, PP_I3_QUERY)] if PP_I3 else [])
if _routes:   # one SRC k/v cache over the kv-heads of every enabled route
    _kv_heads = [h for hs, _ in _routes for h in hs]
    _kv_layers = sorted(heads_per_layer(_kv_heads, "k", layers={l for l, _ in _kv_heads}))
    pp_k = cache_src_proj(pp_src_tok, PP_SRC_POS, _kv_layers, "k")
    pp_v = cache_src_proj(pp_src_tok, PP_SRC_POS, _kv_layers, "v")
pp_restore = layer_outputs(pp_dst_tok, PP_SEG, PP_I2_LAYER_LIST) if PP_I2 else None   # clean DST (padded prompt)


def pp_stack():
    """The enabled interventions for ONE forward / generation (single-use context managers)."""
    s = ExitStack()
    if _routes:
        s.enter_context(headwise_kv_patch_routes(mt, _routes, pp_k, pp_v, PP_DST_POS))
    if PP_I2:
        s.enter_context(residual_patch(pp_restore, PP_SEG, PP_I2_LAYER_LIST))
    if PP_I3 and PP_ALPHA != 1.0:
        s.enter_context(boost_head_ov(mt, GOAL_PATCH_HEADS, PP_ALPHA, positions=(-1,)))
    return s


def pp_iface(mt_, inputs, patches=None, model_kwargs=None):
    with pp_stack():
        return mt_._model(**dict(inputs), **dict(model_kwargs or {}))


# sanity: residual change vs clean DST (every layer) on the I2 window and at the pivot
_all = list(range(PP_L_DST))
_h0 = layer_outputs(pp_dst_tok, _all, list(range(mt.n_layer)))
with pp_stack():
    _h1 = layer_outputs(pp_dst_tok, _all, list(range(mt.n_layer)))
_d = torch.stack([(_h1[l] - _h0[l]).abs().amax(-1) for l in range(mt.n_layer)])     # (layers, T)
rich_print(f"on: I1={PP_I1} I2={PP_I2} I3={PP_I3} · max|Δresid| vs clean DST — task span {_d[:, PP_DST_SPAN[0]:PP_DST_SPAN[1]].max():.3g} · "
           f"I2 window {_d[:, PP_SEG].max():.3g} · pivot {_d[:, -1].max():.3g}")

pp_gen_src = greedy(pp_src_tok, max_new=PP_MAX_NEW)
pp_gen_dst = greedy(pp_dst_tok, max_new=PP_MAX_NEW)
with pp_stack():
    pp_gen_patched = greedy(pp_dst_tok, max_new=PP_MAX_NEW)

pp_ro_src = goalscope_readout("   ↳ goalscope · source_run", pp_src_tok, PP_L_SRC - 1, heads=READOUT_HEADS)[0]["patched_gen"]
pp_ro_dst = goalscope_readout("   ↳ goalscope · destination_run", pp_dst_tok, PP_L_DST - 1, heads=READOUT_HEADS)[0]["patched_gen"]
pp_ro_patched = goalscope_readout("   ↳ goalscope · patched_run", pp_dst_tok, PP_L_DST - 1, heads=READOUT_HEADS,
                                  iface=pp_iface)[0]["patched_gen"]

for tag, col, gen, ro in (("clean SRC (c)", "red1", pp_gen_src, pp_ro_src),
                          ("clean DST (d)", "sea_green3", pp_gen_dst, pp_ro_dst),
                          (f"patched DST · I1={PP_I1} I2={PP_I2} I3={PP_I3}", "salmon1", pp_gen_patched, pp_ro_patched)):
    rich_print(Panel(Text(gen), title=f"[{col}]{escape(tag)}[/{col}] · goalscope @ ' =': [italic]{escape(ro)}[/italic]",
                     title_align="left", border_style="grey42"))
print(f"SRC frontier (c) = {PP_C_DESC!r}\nDST frontier (d) = {PP_D_DESC!r}")

if PP_ORACLE:
    from src.diff.eval import eval_kv_transfer
    _pev = eval_kv_transfer(inst_pp, patched_gen=pp_gen_patched, readouts=(pp_ro_src, pp_ro_dst, pp_ro_patched))
    rich_print(f"[bold]eval · path patching[/bold] : {_pev.summary()}")